## Example 1: Running commands and spawning processes

In [ ]:
import pexpect
import sys
import time

# Example 1: Basic Process Spawning and Interaction
try:
    print("Basic pexpect process spawning and interaction:")
    
    # Show pexpect module information
    print(f"Pexpect version: {getattr(pexpect, '__version__', 'Unknown')}")
    print(f"Pexpect location: {pexpect.__file__}")
    
    # List key pexpect attributes
    key_attrs = ['spawn', 'run', 'EOF', 'TIMEOUT', 'expect', 'sendline']
    available_attrs = [attr for attr in key_attrs if hasattr(pexpect, attr)]
    print(f"Key pexpect attributes: {available_attrs}")
    
    # Basic command execution with pexpect.run
    print(f"\n--- Basic Command Execution ---")
    
    # Simple command execution
    try:
        result = pexpect.run('echo "Hello from pexpect"')
        print(f"Echo result: {result.decode().strip()}")
        assert result.decode().strip() == "Hello from pexpect", result
    except Exception as e:
        print(f"Echo command error: {e}")
        raise
    
    # Command with timeout
    try:
        result = pexpect.run('date', timeout=5)
        print(f"Date result: {result.decode().strip()}")
    except Exception as e:
        print(f"Date command error: {e}")
        raise
    
    # Test platform-specific commands
    print(f"\n--- Platform-Specific Commands ---")
    
    platform_commands = []
    if sys.platform.startswith('darwin') or sys.platform.startswith('linux'):
        platform_commands = [
            ('whoami', "Get current user"),
            ('pwd', "Get current directory"),
            ('ls -la /tmp', "List temp directory"),
        ]
    elif sys.platform.startswith('win'):
        platform_commands = [
            ('echo %USERNAME%', "Get current user"),
            ('cd', "Get current directory"),
        ]
    
    for cmd, description in platform_commands:
        try:
            result = pexpect.run(cmd, timeout=10)
            output = result.decode().strip()[:100]  # Limit output
            print(f"{description}: {output}")
        except Exception as cmd_error:
            print(f"{description} error: {cmd_error}")
            raise
    
    # Show pexpect exceptions
    print(f"\n--- Pexpect Exceptions ---")
    
    exception_types = [
        'EOF', 'TIMEOUT', 'ExceptionPexpect', 
        'spawn', 'before', 'after'
    ]
    
    for exc_name in exception_types:
        if hasattr(pexpect, exc_name):
            exc_type = getattr(pexpect, exc_name)
            print(f"  {exc_name}: {type(exc_type)}")
    
    # Basic spawn example (safe command)
    print(f"\n--- Basic Spawn Example ---")
    
    try:
        # Spawn a simple command
        child = pexpect.spawn('echo "Interactive test"')
        child.expect(pexpect.EOF, timeout=5)
        output = child.before.decode() if child.before else ""
        print(f"Spawn output: {output.strip()}")
        assert output.strip() == "Interactive test", output
        child.close()
        print(f"Exit status: {child.exitstatus}")
        assert child.exitstatus == 0, child.exitstatus
        
    except pexpect.TIMEOUT:
        raise AssertionError(f"pexpect timed out: {child.before!r}") from None
    except Exception as spawn_error:
        print(f"Spawn error: {spawn_error}")
        raise
    
    print("Example 1: Basic process spawning completed")
    
except Exception as e:
    print(f"Error in basic pexpect operations: {e}")
    raise AssertionError(
        f"pexpect example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Driving interactive programs

In [ ]:
import pexpect
import sys
import tempfile

# Example 2: Interactive Process Communication
try:
    print("Interactive process communication with pexpect:")
    
    # Interactive Python session example
    print("\n--- Interactive Python Session ---")
    
    try:
        # Start a Python interpreter
        child = pexpect.spawn(sys.executable, ['-i'], timeout=10)
        
        # Wait for Python prompt
        child.expect('>>> ')
        print("Python interpreter started")
        
        # Send some commands
        commands = [
            'x = 42',
            'y = 28', 
            'result = x + y',
            'print(f"Result: {result}")'
        ]
        
        outputs = []
        for cmd in commands:
            child.sendline(cmd)
            child.expect('>>> ')
            output = child.before.decode().strip()
            outputs.append(output)
            if output and not output.startswith(cmd):
                print(f"  {cmd} -> {output}")
            else:
                print(f"  {cmd} -> executed")
        assert "Result: 70" in outputs[-1], outputs[-1]
        
        # Arithmetic in the child interpreter: match the exact printed result
        child.sendline('print((10 + 5 - 3) * 7)')
        child.expect_exact('84')
        child.expect('>>> ')
        print("  print((10 + 5 - 3) * 7) -> 84")
        
        # Exit Python
        child.sendline('exit()')
        child.expect(pexpect.EOF)
        child.close()
        print("Python session completed")
        
    except pexpect.TIMEOUT:
        raise AssertionError(f"pexpect timed out: {child.before!r}") from None
    except Exception as py_error:
        print(f"Python session error: {py_error}")
        raise
    
    # File interaction example
    print("\n--- File Interaction Example ---")
    
    # Create a temporary script
    temp_script = tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False)
    script_content = '''
import sys
print("Interactive script started")
while True:
    try:
        line = input("Enter command (quit to exit): ")
        if line.strip() == 'quit':
            print("Goodbye!")
            break
        elif line.strip() == 'hello':
            print("Hello there!")
        elif line.startswith('echo '):
            print(f"Echo: {line[5:]}")
        else:
            print(f"Unknown command: {line}")
    except EOFError:
        break
'''
    
    temp_script.write(script_content)
    temp_script.close()
    
    try:
        # Run the interactive script
        child = pexpect.spawn(sys.executable, [temp_script.name], timeout=10)
        
        # Wait for initial prompt
        child.expect('Enter command.*: ')
        print("Interactive script started")
        
        # Send test commands
        test_inputs = ['hello', 'echo test message', 'quit']
        replies = []
        
        for input_cmd in test_inputs:
            child.sendline(input_cmd)
            if input_cmd != 'quit':
                child.expect('Enter command.*: ')
            else:
                child.expect('Goodbye!')
            
            output = child.before.decode().strip()
            lines = output.split('\n')
            if lines:
                last_line = lines[-1].strip()
                if last_line and not last_line.startswith('Enter command'):
                    print(f"  > {input_cmd}")
                    print(f"  < {last_line}")
                    replies.append(last_line)
        
        assert replies[:2] == ["Hello there!", "Echo: test message"], replies
        child.expect(pexpect.EOF)
        child.close()
        assert child.exitstatus == 0, child.exitstatus
        print("Script interaction completed")
        
    except Exception as script_error:
        print(f"Script interaction error: {script_error}")
        raise
    finally:
        # Cleanup
        import os
        os.unlink(temp_script.name)
    
    print("Example 2: Interactive process communication completed")
    
except Exception as e:
    print(f"Error in interactive communication: {e}")
    raise AssertionError(
        f"pexpect example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Pattern matching with expect

In [ ]:
import os
import pexpect
import re
import sys

# Example 3: Pattern Matching and Expect Operations
try:
    print("Pattern matching and expect operations:")
    
    # Pattern matching examples
    print("\n--- Pattern Matching Examples ---")
    
    # Create a simple interactive command for testing
    def create_test_command():
        """Create a test script that outputs predictable patterns"""
        import tempfile
        import os
        
        script_content = '''#!/usr/bin/env python3
import sys
import time

print("Test program started")
print("Type 'help' for commands")

while True:
    try:
        cmd = input("test> ")
        if cmd == 'quit':
            print("Exiting...")
            break
        elif cmd == 'help':
            print("Commands: status, error, wait, quit")
        elif cmd == 'status':
            print("Status: OK")
        elif cmd == 'error':
            print("ERROR: Something went wrong")
        elif cmd == 'wait':
            print("Waiting...")
            time.sleep(1)
            print("Done waiting")
        else:
            print(f"Unknown command: {cmd}")
    except EOFError:
        break
'''
        
        temp_file = tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False)
        temp_file.write(script_content)
        temp_file.close()
        
        return temp_file.name
    
    # Test various expect patterns
    print("Testing expect patterns:")
    
    script_path = create_test_command()
    try:
        # Start the test program
        child = pexpect.spawn(sys.executable, [script_path], timeout=15)
        
        # Test 1: Simple string pattern
        child.expect("Type 'help' for commands")
        print("✓ Simple string pattern matched")
        
        # Test 2: Regular expression pattern
        child.expect("test> ")
        print("✓ Prompt pattern matched")
        
        # Send command and expect response
        child.sendline("status")
        child.expect("Status: OK")
        print("✓ Command response matched")
        
        # Test 3: Multiple patterns with expect
        child.expect("test> ")
        child.sendline("help")
        
        # Expect one of multiple patterns
        patterns = ["Commands:", "Help:", "Usage:"]
        index = child.expect(patterns)
        print(f"✓ Multiple pattern match (index {index}): {patterns[index]}")
        
        # Test 4: Timeout handling
        child.expect("test> ")
        child.sendline("wait")
        
        try:
            child.expect("Done waiting", timeout=5)
            print("✓ Timeout pattern matched within time limit")
        except pexpect.TIMEOUT:
            raise AssertionError(f"pexpect timed out: {child.before!r}") from None
        
        # Test 5: Error pattern handling
        child.expect("test> ")
        child.sendline("error")
        child.expect("ERROR:")
        print("✓ Error pattern matched")
        
        # Clean exit
        child.expect("test> ")
        child.sendline("quit")
        child.expect("Exiting...")
        child.expect(pexpect.EOF)
        child.close()
        
        print("Test program interaction completed")
        
    except Exception as pattern_error:
        print(f"Pattern testing error: {pattern_error}")
        raise
    finally:
        # Cleanup
        os.unlink(script_path)
    
    # Regular expression pattern examples
    print("\n--- Regular Expression Patterns ---")
    
    regex_patterns = [
        (r'\d+', "Matches one or more digits"),
        (r'[A-Z][a-z]+', "Matches capitalized words"),
        (r'\w+@\w+\.\w+', "Matches basic email pattern"),
        (r'ERROR: .*', "Matches error messages"),
        (r'Status: (OK|FAIL|PENDING)', "Matches status with capture group")
    ]
    
    test_strings = [
        "Status: OK",
        "ERROR: File not found",
        "User123 logged in",
        "Email: user@example.com",
        "Processing 42 items"
    ]
    
    for pattern, description in regex_patterns:
        print(f"\nPattern: {pattern} - {description}")
        compiled = re.compile(pattern)
        
        for test_str in test_strings:
            match = compiled.search(test_str)
            if match:
                print(f"  ✓ '{test_str}' -> {match.group()}")
    
    # Advanced expect features
    print("\n--- Advanced Expect Features ---")
    
    expect_features = [
        "expect(pattern, timeout=30) - Set custom timeout",
        "expect([pat1, pat2, pat3]) - Multiple patterns", 
        "expect(pexpect.EOF) - End of file",
        "expect(pexpect.TIMEOUT) - Explicit timeout handling",
        "child.before - Text before match",
        "child.after - Text that matched",
        "child.match - Match object for regex patterns"
    ]
    
    for feature in expect_features:
        print(f"  • {feature}")
    
    print("Example 3: Pattern matching and expect operations completed")
    
except Exception as e:
    print(f"Error in pattern matching: {e}")
    raise AssertionError(
        f"pexpect example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Process control and signals

In [ ]:
import pexpect
import sys
import time

# Example 4: Process Control and Signal Handling
try:
    print("Process control and signal handling:")
    
    # Long-running process simulation
    print("\n--- Process Control Examples ---")
    
    # Test process termination
    try:
        child = pexpect.spawn('sleep', ['30'], timeout=10)
        
        # Let it run briefly
        time.sleep(1)
        
        # Check if process is alive
        print(f"Process alive: {child.isalive()}")
        print(f"Process PID: {child.pid}")
        assert child.isalive(), "sleep 30 exited on its own"
        
        # Terminate the process
        assert child.terminate(force=False), "terminate() did not stop the process"
        
        # Wait for termination
        child.expect(pexpect.EOF, timeout=5)
        child.close()
        print(f"Process terminated gracefully")
        print(f"Exit status: {child.exitstatus}")
        print(f"Signal status: {child.signalstatus}")
        assert child.exitstatus is None and child.signalstatus is not None, (child.exitstatus, child.signalstatus)
        
    except pexpect.TIMEOUT:
        print("Process termination timed out")
        child.terminate(force=True)
        raise
    except Exception as proc_error:
        print(f"Process control error: {proc_error}")
        raise
    
    # Command execution with different methods
    print("\n--- Execution Methods Comparison ---")
    
    test_command = 'echo "Test output"'
    
    # Method 1: pexpect.run (simple)
    try:
        result1 = pexpect.run(test_command)
        print(f"run() result: {result1.decode().strip()}")
    except Exception as e:
        print(f"run() error: {e}")
        raise
    
    # Method 2: spawn with expect
    try:
        child = pexpect.spawn(test_command)
        child.expect(pexpect.EOF)
        result2 = child.before.decode().strip() if child.before else ""
        print(f"spawn() result: {result2}")
        child.close()
    except Exception as e:
        print(f"spawn() error: {e}")
        raise
    
    # Error handling examples
    print("\n--- Error Handling Examples ---")
    
    # Test invalid command: spawn() itself refuses a command it cannot find on PATH
    try:
        child = pexpect.spawn('nonexistent_command_xyz')
    except pexpect.exceptions.ExceptionPexpect as e:
        assert 'nonexistent_command_xyz' in str(e), e
        print(f"Expected error with invalid command: {type(e).__name__}")
    else:
        child.close(force=True)
        raise AssertionError("pexpect.spawn accepted a command that does not exist")
    
    print("Example 4: Process control and signal handling completed")
    
except Exception as e:
    print(f"Error in process control: {e}")
    raise AssertionError(
        f"pexpect example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: A shell session with a fixed prompt

In [ ]:
import os
import tempfile

import pexpect

# Example 5: Practical pexpect Applications
try:
    print("Practical pexpect applications:")
    
    # Start /bin/sh with a prompt we choose. expect_exact() then waits for that fixed string,
    # whatever prompt the user's uid or shell configuration would otherwise give.
    PROMPT = "PEXPECT> "
    shell_env = {key: value for key, value in os.environ.items() if key != "ENV"}
    shell_env.update(PS1=PROMPT, TERM="dumb")

    def start_shell(cwd):
        shell = pexpect.spawn("/bin/sh", cwd=cwd, env=shell_env, encoding="utf-8",
                              timeout=10, dimensions=(24, 500))
        shell.expect_exact(PROMPT)
        return shell

    def run_in_shell(shell, command):
        """Send one command; return its output (the text between the echoed command and the next prompt)."""
        shell.sendline(command)
        shell.expect_exact(PROMPT)
        echoed, _, output = shell.before.replace("\r\n", "\n").partition("\n")
        return output.strip()

    with tempfile.TemporaryDirectory() as workdir:
        print("\n--- Session Management ---")
        shell = start_shell(workdir)
        commands = [
            ("echo $0", "Show shell type"),
            ("pwd", "Show current directory"),
            ('echo "Hello from shell"', "Test echo"),
            ("expr 2 + 3", "Simple calculation"),
        ]
        results = {}
        for cmd, desc in commands:
            results[cmd] = run_in_shell(shell, cmd)
            print(f"  {desc}: {results[cmd]}")
        assert results["echo $0"].endswith("sh"), results["echo $0"]
        assert os.path.samefile(results["pwd"], workdir), results["pwd"]
        assert results['echo "Hello from shell"'] == "Hello from shell", results
        assert results["expr 2 + 3"] == "5", results

        print("\n--- File Operations ---")
        test_file = os.path.join(workdir, "pexpect_test.txt")
        assert run_in_shell(shell, 'echo "Test content for pexpect" > pexpect_test.txt') == ""
        assert run_in_shell(shell, "cat pexpect_test.txt") == "Test content for pexpect"
        with open(test_file) as f:
            assert f.read() == "Test content for pexpect\n"
        print("✓ File created through the shell and read back")
        assert run_in_shell(shell, "rm pexpect_test.txt") == ""
        assert not os.path.exists(test_file)
        print("✓ File removed through the shell")

        shell.sendline("exit")
        shell.expect(pexpect.EOF)
        shell.close()
        assert shell.exitstatus == 0, (shell.exitstatus, shell.signalstatus)
    
    # Show pexpect module info
    print(f"\nPexpect module info:")
    print(f"Version: {getattr(pexpect, '__version__', 'Unknown')}")
    print(f"Location: {pexpect.__file__}")
    
    # Key pexpect constants and exceptions
    key_items = ['EOF', 'TIMEOUT', 'spawn', 'run', 'expect']
    available = [item for item in key_items if hasattr(pexpect, item)]
    print(f"Key components: {', '.join(available)}")
    
    print("Example 5: Practical pexpect applications completed")
    
except Exception as e:
    print(f"Error in practical applications: {e}")
    raise AssertionError(
        f"pexpect example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e